In [1]:
from pathlib import Path

ROOT = Path.cwd().parent
DATA = ROOT / "data"
RANDOM_STATE = 42

In [2]:
import numpy as np
import pandas as pd
import timm
import torch
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm

MODEL_NAME = "convnext_base.fb_in22k_ft_in1k_384"
BATCH_SIZE = 16
EMBEDDINGS_DIR = DATA / "embeddings"
DEVICE = torch.device("mps")

torch.manual_seed(RANDOM_STATE)

/Users/dmytro_lazarenko/PycharmProjects/deep-learning-assignment2/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# num_classes=0 removes the last layer, so the model returns the 1024 pooled features
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0).eval().to(DEVICE)
data_config = timm.data.resolve_data_config({}, model=model)
to_normalized_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=data_config["mean"], std=data_config["std"]),
])
data_config["mean"], data_config["std"]

((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

In [4]:
def load_batch(image_dir: Path, image_ids: list[str]) -> torch.Tensor:
    """Load images as one normalized tensor. No resize, so all images must have the same size."""
    images = []
    for image_id in image_ids:
        with Image.open(image_dir / f"{image_id}.jpg") as image:
            images.append(to_normalized_tensor(image.convert("RGB")))
    return torch.stack(images)


@torch.inference_mode()
def embed_images(image_dir: Path, image_ids: list[str]) -> np.ndarray:
    """Return one embedding row per image id, in the same order."""
    batches = []
    for start in tqdm(range(0, len(image_ids), BATCH_SIZE)):
        batch = load_batch(image_dir, image_ids[start:start + BATCH_SIZE])
        batches.append(model(batch.to(DEVICE)).cpu().numpy())
    return np.concatenate(batches)


def save_split_embeddings(split: str) -> None:
    """Embed every image of the split and save the array to EMBEDDINGS_DIR/{split}.npy."""
    image_ids = pd.read_csv(DATA / f"{split}.csv")["image_id"].tolist()
    EMBEDDINGS_DIR.mkdir(exist_ok=True)
    np.save(EMBEDDINGS_DIR / f"{split}.npy", embed_images(DATA / split, image_ids))

In [5]:
# Smoke test on a few images; nothing is saved
sample_ids = pd.read_csv(DATA / "train.csv")["image_id"].tolist()[:5]
sample_embeddings = embed_images(DATA / "train", sample_ids)
print(sample_embeddings.shape, sample_embeddings.dtype, np.isfinite(sample_embeddings).all())
sample_embeddings[:, :5]

100%|██████████| 1/1 [00:00<00:00,  1.83it/s]

(5, 1024) float32 True


array([[-0.53038484,  0.83303064,  0.55366755,  0.7811622 , -0.25470862],
       [ 0.437998  ,  0.6642992 ,  0.59374106,  0.86405915, -0.465116  ],
       [-0.8291849 ,  1.7303973 ,  2.2988493 , -0.23909542,  0.56684035],
       [-0.93538   ,  0.8820587 , -0.09786308,  0.9376785 , -0.33089024],
       [ 0.07030281,  0.31795335, -1.2461593 , -0.32264596,  0.48937058]],
      dtype=float32)

In [6]:
for split in ("train", "test"):
    save_split_embeddings(split)

100%|██████████| 1168/1168 [11:18<00:00,  1.72it/s]


In [7]:
def describe_saved_embeddings(splits: tuple[str, ...]) -> pd.DataFrame:
    """Return one row per split with the shape of its saved array."""
    shapes = {split: np.load(EMBEDDINGS_DIR / f"{split}.npy").shape for split in splits}
    return pd.DataFrame(
        [(split, rows, columns) for split, (rows, columns) in shapes.items()],
        columns=["split", "rows", "columns"],
    )


describe_saved_embeddings(("train", "test"))

,split,rows,columns
0,train,12260,1024
1,test,18673,1024


In [8]:
def name_label_combination(label_columns: list[str], labels: tuple) -> str:
    """Join the names of the active labels, e.g. "Cat + Moris"."""
    return " + ".join(c for c, value in zip(label_columns, labels) if value == 1) or "No labels"


train = pd.read_csv(DATA / "train.csv")
label_columns = [column for column in train.columns if column != "image_id"]
combination_names = pd.Series(
    [name_label_combination(label_columns, labels) for labels in train[label_columns].itertuples(index=False)],
    name="combination",
)
train_embeddings = np.load(EMBEDDINGS_DIR / "train.npy")

In [9]:
def find_nearest_neighbor_indices(embeddings: np.ndarray) -> np.ndarray:
    """Return, for each row, the index of the most cosine-similar other row."""
    unit_embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    similarities = unit_embeddings @ unit_embeddings.T
    np.fill_diagonal(similarities, -np.inf)
    return similarities.argmax(axis=1)


def summarize_neighbor_agreement(combination_names: pd.Series, neighbor_indices: np.ndarray) -> pd.DataFrame:
    """Return one row per combination: photo count and % whose nearest photo has the same combination."""
    has_same_combination = combination_names.to_numpy() == combination_names.to_numpy()[neighbor_indices]
    return (
        pd.DataFrame({"combination": combination_names, "same": has_same_combination})
        .groupby("combination")["same"]
        .agg(photos="size", same_combination_pct=lambda same: round(100 * same.mean(), 1))
        .sort_values("photos", ascending=False)
        .reset_index()
    )

In [10]:
neighbor_indices = find_nearest_neighbor_indices(train_embeddings)
summarize_neighbor_agreement(combination_names, neighbor_indices)

,combination,photos,same_combination_pct
0,Cat,4989,99.5
1,Dog,4984,99.6
2,No labels,2162,99.8
3,Cat + Moris,42,78.6
4,Dog + Biatrix,37,100.0
5,Cat + Motya,32,81.2
6,Cat + Moris + Motya,10,30.0
7,Cat + Dog + Moris + Biatrix,4,75.0


## Conclusions

**Model choice**
- ConvNeXt is a strong CNN of recent years. It is pretrained on ImageNet, so the assignment allows it.
- ConvNeXt has several sizes. I chose `convnext_base` as a balance between top-1 accuracy and the number of parameters.
- This version was trained on 384x384 images, which is close to our 450x375 images. We use the images without resize or crop, so we lose very little information.

**Embeddings**
- We made the embeddings (1024 numbers per image) only to run the similarity check: the nearest train photo by cosine similarity, skipping the photo itself.
- The embeddings look healthy. The 3 majority classes (Cat, Dog, No labels) have 99+% of nearest photos with the same combination.

**Majority classes**
- No labels is the highest (99.8%). Its photos are far from cats and dogs, so even if a lobster is matched with a horse, the combination is still correct.

**Named pets**
- Moris and Motya give a not bad result (79% and 81%). Probably the embeddings catch the home or the environment of the photos. This cause is not confirmed.
- Biatrix is very high: all 37 photos (100%) have the same combination as the nearest photo. The reason is not clear. Maybe the photos are very similar to each other, or maybe the dog looks unique, so photos with a similar background are not close to it.
- Cat + Moris + Motya (30%, 10 photos) is the weakest. Probably the nearest photo is often one of the two cats alone, which counts as a wrong combination. With 10 photos, one photo changes the result by 10 points.
